# Lite-DETR — 범용 CCTV 사람 탐지 (Colab GPU)

- **기본 입력:** Google Drive `data/videos`의 로컬 영상 (AVI/MP4/MOV/MKV)
- **선택 입력:** MEVA 공개 S3 객체 Key를 지정하면 필요한 영상만 다운로드
- **탐지:** 사람(person, COCO class_id=1), 200ms, confidence 0.10, NMS 0.50, 원본 픽셀 xyxy
- **실행:** `SMOKE_TEST=True`면 영상당 3프레임만 처리, `False`면 전체 처리. 출력 디렉터리를 구분해 기존 T3 및 이전 smoke 결과 보존
- **검증 상태:** G301(사람 0건), G330(3프레임 229 탐지 후보)에서 smoke test 완료. 전체 동영상/ByteTrack/B3-B4 연동은 별도 검증 필요

**실행 순서:** 아래 셀을 위에서 아래로 차례대로 실행하세요. Colab GPU 런타임 필요. 원본 영상·가중치는 GitHub에 넣지 않습니다.


## 1. Google Drive 연결

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Colab 의존성 설치 (import av 전에 실행)

In [ ]:
%pip install -q av PyYAML ninja cython scipy termcolor addict yapf pycocotools fvcore iopath ultralytics requests


## 3. 사용자 설정 (영상 폴더·S3·smoke test)

In [ ]:
from pathlib import Path
import os, sys, json, time, gc, re, shutil, subprocess, urllib.request, hashlib
from urllib.parse import quote
from collections import Counter
import requests
import numpy as np
import torch
import av

# ── 사용자가 변경할 기본 설정 ──
PROJECT = Path('/content/drive/MyDrive/CCTV_Project')
VIDEO_DIR = PROJECT / 'data/videos'
MANIFEST = None                      # 예: PROJECT / 'data/manifest.jsonl'
VIDEO_IDS = None                     # None=폴더의 모든 영상, 목록이면 해당 stem/video_id만

# S3 사용은 선택 사항. GitHub 기본값은 False (자동 대용량 다운로드 방지)
DOWNLOAD_FROM_S3 = False
S3_BASE = 'https://mevadata-public-01.s3.amazonaws.com/'
S3_KEYS = []                         # 예: 'drop-4-hadcv22/2018-03-10/10/...G330.r13.avi'

STEP_MS = 200
MAX_PTS_ERROR_MS = 100
CONF_THRESHOLD = 0.10
NMS_IOU = 0.50
SMOKE_TEST = True                    # False=영상 전체 추론
SMOKE_FRAMES = 3

# ── 출력·모델 설정 (T3 기본 모델/전처리 보존) ──
OUT = PROJECT / 'runs/b_litedetr_generic'
CLASSES = {1: 'person'}
EXTENSIONS = {'.avi', '.mp4', '.mov', '.mkv', '.m4v'}
ROOT = Path('/content/cctv-litedetr')
REPO = ROOT / 'Lite-DETR'
OPS = REPO / 'models/dino/ops'
CHECKPOINT_NAME = 'r50_s2ex3_49.9.pth'

OUT.mkdir(parents=True, exist_ok=True)
VIDEO_DIR.mkdir(parents=True, exist_ok=True)
assert STEP_MS > 0 and 0 <= CONF_THRESHOLD <= 1
assert SMOKE_FRAMES >= 1
assert torch.cuda.is_available(), 'Colab GPU 런타임으로 변경해야 합니다.'
print('GPU:', torch.cuda.get_device_name(0))
print('VIDEO_DIR:', VIDEO_DIR)
print('S3 자동 다운로드:', DOWNLOAD_FROM_S3)


## 4. 선택적 S3 영상 다운로드 (기본적으로 실행해도 다운로드하지 않음)

In [ ]:
# 선택적 S3 다운로드. DOWNLOAD_FROM_S3=False면 로컬 영상만 사용합니다.
def download_s3_video(s3_key: str) -> Path:
    key = str(s3_key).strip().lstrip('/')
    if not key or Path(key).suffix.lower() not in EXTENSIONS:
        raise ValueError(f'지원하지 않는 S3 영상 Key: {s3_key!r}')
    if any(part in ('.', '..') for part in key.split('/')):
        raise ValueError('경로 이동 문자는 사용할 수 없습니다.')
    filename = Path(key).name
    destination = VIDEO_DIR / filename
    if destination.is_file() and destination.stat().st_size > 0:
        print('SKIP 기존 영상:', destination.name)
        return destination
    url = S3_BASE.rstrip('/') + '/' + quote(key, safe='/')
    temporary = VIDEO_DIR / (filename + '.download')
    if temporary.exists():
        raise FileExistsError(f'미완료 다운로드 파일 먼저 확인: {temporary}')
    print('DOWNLOAD:', filename)
    try:
        with requests.get(url, stream=True, timeout=(15, 120)) as response:
            response.raise_for_status()
            length = response.headers.get('Content-Length')
            expected = int(length) if length is not None else None
            with temporary.open('xb') as f:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk: f.write(chunk)
        actual = temporary.stat().st_size
        if actual == 0 or (expected is not None and actual != expected):
            raise RuntimeError(f'다운로드 크기 오류: expected={expected}, actual={actual}')
        # 이름이 같은 파일을 다른 S3 Key가 가리킬 수 있으므로 자동 덮어쓰기 금지
        if destination.exists():
            raise FileExistsError(f'동일 파일명 충돌: {destination}')
        temporary.rename(destination)
        print(f'PASS: {destination.name} ({actual / 1024**2:.2f} MiB)')
        return destination
    except Exception:
        # 실패 시 재시도를 허용하되 기존 정상 파일은 보존
        temporary.unlink(missing_ok=True)
        raise

if DOWNLOAD_FROM_S3:
    if not S3_KEYS:
        raise ValueError('DOWNLOAD_FROM_S3=True이면 S3_KEYS를 입력하세요.')
    for key in S3_KEYS:
        download_s3_video(key)
else:
    print('S3 다운로드 생략 — 기존 Drive 영상 사용')

local_videos = sorted(p for p in VIDEO_DIR.rglob('*') if p.is_file() and p.suffix.lower() in EXTENSIONS)
print('Drive에서 찾은 영상 수:', len(local_videos))
for p in local_videos[:20]:
    print(' -', p.name, f'{p.stat().st_size / 1024**2:.1f} MiB')
if len(local_videos) > 20: print(' ... 그 외',len(local_videos)-20,'개')


## 5. 영상 자동 검색 / manifest 로딩 / 메타데이터 점검

In [ ]:
def safe_id(value):
    value = str(value).strip()
    if not value or value in ('.', '..') or not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9._-]*', value):
        raise ValueError(f'video_id에 허용되지 않는 문자: {value!r}')
    return value


def load_sources():
    if MANIFEST is None:
        if not VIDEO_DIR.is_dir():
            raise FileNotFoundError(f'영상 폴더 없음: {VIDEO_DIR}')
        paths = sorted(p for p in VIDEO_DIR.rglob('*') if p.is_file() and p.suffix.lower() in EXTENSIONS)
        if not paths:
            raise FileNotFoundError(f'처리할 영상 없음: {VIDEO_DIR}')
        rows = [{'video_id': safe_id(p.stem), 'path': p} for p in paths]
    else:
        manifest = Path(MANIFEST)
        if not manifest.is_file():
            raise FileNotFoundError(f'manifest 없음: {manifest}')
        rows = []
        for lineno, line in enumerate(manifest.read_text(encoding='utf-8').splitlines(), 1):
            if not line.strip(): continue
            raw = json.loads(line)
            vid = safe_id(raw['video_id'])
            # path/video_path/filename 중 하나를 제공. 경로가 없다면 VIDEO_DIR에서 영상 stem으로 찾음.
            spec = raw.get('video_path') or raw.get('path') or raw.get('filename')
            if spec:
                p = Path(spec).expanduser()
                if not p.is_absolute(): p = VIDEO_DIR / p
            else:
                matches = [p for p in VIDEO_DIR.rglob('*') if p.is_file() and p.suffix.lower() in EXTENSIONS and (p.stem == vid or p.stem == vid.removeprefix('meva_') + '.r13')]
                if len(matches) != 1: raise ValueError(f'{lineno}행 {vid}: 영상 경로 모호/없음 ({matches})')
                p = matches[0]
            rows.append({'video_id': vid, 'path': p, 'meta': raw})
    if VIDEO_IDS is not None:
        selected = set(VIDEO_IDS)
        actual = {r['video_id'] for r in rows}
        if selected - actual: raise ValueError(f'VIDEO_IDS 중 없는 ID: {sorted(selected - actual)}')
        rows = [r for r in rows if r['video_id'] in selected]
    if len({r['video_id'] for r in rows}) != len(rows):
        raise ValueError('중복 video_id가 있습니다. manifest에서 각각 다른 ID를 지정하세요.')
    if not rows: raise ValueError('선택된 영상이 없습니다.')
    return rows


def probe_video(path):
    if not path.is_file(): raise FileNotFoundError(path)
    with av.open(str(path)) as container:
        stream = container.streams.video[0]
        width, height = stream.width, stream.height
        # container.duration은 마이크로초; 없으면 stream.duration/time_base 사용
        if stream.duration is not None and stream.time_base is not None:
            duration_ms = round(float(stream.duration * stream.time_base) * 1000)
        elif container.duration is not None:
            duration_ms = round(float(container.duration) / 1000)
        else:
            raise ValueError(f'{path.name}: 길이 정보를 읽을 수 없음. manifest에 available_end_ms 지정 필요')
    if duration_ms <= 0: raise ValueError(f'{path}: duration_ms={duration_ms}')
    return int(width), int(height), duration_ms


VIDEO_CONFIGS = []
for source in load_sources():
    vid, path = source['video_id'], Path(source['path'])
    meta = source.get('meta', {})
    width, height, measured_end = probe_video(path)
    if 'width' in meta and int(meta['width']) != width: raise ValueError(f'{vid}: width 불일치')
    if 'height' in meta and int(meta['height']) != height: raise ValueError(f'{vid}: height 불일치')
    begin = int(meta.get('available_start_ms', 0))
    end = int(meta.get('available_end_ms', measured_end))
    if begin < 0 or end <= begin: raise ValueError(f'{vid}: 잘못된 시간 범위 {begin}~{end}')
    if end > measured_end + 200: raise ValueError(f'{vid}: manifest 종료 시각이 영상 길이 초과 ({end}>{measured_end})')
    # 출력 파일은 재실행 시 이어 쓰기를 위해 동일한 ID 유지
    timestamps = list(range(begin, end, STEP_MS))
    if not timestamps: raise ValueError(f'{vid}: 샘플링 대상 없음')
    VIDEO_CONFIGS.append(dict(video_id=vid, path=path,
        output=OUT / f'detections_{vid}_pts.jsonl', timestamps=timestamps,
        total=len(timestamps), start_ms=begin, end_ms=end,
        offset_ms=int(meta.get('source_offset_ms') or 0),
        width=width, height=height))

print('선택 영상:', len(VIDEO_CONFIGS), '전체 요청 프레임:', sum(j['total'] for j in VIDEO_CONFIGS))
for j in VIDEO_CONFIGS:
    print(j['video_id'], j['total'], 'frames', j['width'], 'x', j['height'], '->', j['output'])

## 6. Lite-DETR 공식 저장소·체크포인트 준비

In [ ]:
ROOT.mkdir(parents=True, exist_ok=True)
if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/IDEA-Research/Lite-DETR.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '6392353'], check=True)
assert (REPO / 'config/DINO/DINO_4scale.py').is_file()

# Colab /content 는 런타임 재시작 시 초기화됨. Drive 캐시가 있다면 우선 사용.
DRIVE_CKPT = PROJECT / 'checkpoints' / CHECKPOINT_NAME
LOCAL_CKPT = ROOT / 'checkpoints' / CHECKPOINT_NAME
LOCAL_CKPT.parent.mkdir(parents=True, exist_ok=True)
if not LOCAL_CKPT.is_file() or LOCAL_CKPT.stat().st_size < 100_000_000:
    if DRIVE_CKPT.is_file() and DRIVE_CKPT.stat().st_size > 100_000_000:
        shutil.copy2(DRIVE_CKPT, LOCAL_CKPT)
        print('Drive 체크포인트 캐시 재사용')
    else:
        url = 'https://github.com/IDEA-Research/Lite-DETR/releases/download/litedetr/r50_s2ex3_49.9.pth'
        tmp = LOCAL_CKPT.with_suffix('.pth.download')
        urllib.request.urlretrieve(url, tmp)
        assert tmp.stat().st_size > 100_000_000, '체크포인트 크기 이상'
        tmp.replace(LOCAL_CKPT)
        print('공식 체크포인트 다운로드')
assert LOCAL_CKPT.stat().st_size > 100_000_000
print('PASS: 저장소 + 체크포인트', round(LOCAL_CKPT.stat().st_size/1024**2, 1), 'MiB')

## 7. CUDA 확장 빌드 (T4 환경)

In [ ]:
CUDA_FILE = OPS / 'src/cuda/ms_deform_attn_cuda.cu'
src = CUDA_FILE.read_text(encoding='utf-8')
# 실수로 전체 value.type()을 치환했던 경우까지 복구
src = src.replace('value.scalar_type().is_cuda()', 'value.is_cuda()')
pattern = re.compile(r'(AT_DISPATCH_FLOATING_TYPES(?:_AND_HALF)?\s*\(\s*)value\.type\(\)')
# 컴파일 매크로 안의 값 2곳만 변경
patched, count = pattern.subn(r'\g<1>value.scalar_type()', src)
if count not in (0, 2):
    raise RuntimeError(f'CUDA API 패치 예상과 다른 구조: {count}곳')
assert patched.count('AT_DISPATCH_FLOATING_TYPES(value.scalar_type()') == 2, '두 DISPATCH 매크로 검사 실패'
assert 'value.scalar_type().is_cuda()' not in patched
if patched != CUDA_FILE.read_text(encoding='utf-8'):
    shutil.copy2(CUDA_FILE, CUDA_FILE.with_suffix('.cu.original.bak'))
    CUDA_FILE.write_text(patched, encoding='utf-8')

# 기존 컴파일 바이너리는 재시작 시 사라짐. 현재 Python ABI용 빌드 여부를 확인.
existing = list(OPS.glob('MultiScaleDeformableAttention*.so'))
if not existing:
    env = os.environ.copy()
    env.update({'TORCH_CUDA_ARCH_LIST': '7.5', 'MAX_JOBS': '2', 'CUDA_HOME': '/usr/local/cuda'})
    log = ROOT / 'cuda_build_recovery.log'
    with log.open('w', encoding='utf-8') as f:
        build = subprocess.run([sys.executable, 'setup.py', 'build_ext', '--inplace'],
            cwd=str(OPS), env=env, stdout=f, stderr=subprocess.STDOUT)
    if build.returncode != 0:
        lines = log.read_text(encoding='utf-8', errors='replace').splitlines()
        hits = [i for i,s in enumerate(lines) if re.search(r'error:|fatal error:|FAILED:|killed|ninja: build stopped', s, re.I)]
        for i in hits[:6]:
            print('\n'.join(lines[max(i-2,0):i+5]));print('---')
        raise RuntimeError(f'CUDA 컴파일 실패, 로그: {log}')

sys.path.insert(0, str(OPS))
import MultiScaleDeformableAttention as MSDA
assert hasattr(MSDA, 'ms_deform_attn_forward')
print('PASS: CUDA 확장 빌드 및 import:', MSDA.__file__)
# 공식 전체 test.py는 수치 gradient 검사 중 T4 OOM 사례가 있어 재실행하지 않음.

## 8. Lite-DETR 모델 로딩 및 전처리

In [ ]:
os.chdir(REPO)
sys.path.insert(0, str(REPO))
from util.slconfig import SLConfig
from main import get_args_parser, build_model_main
from datasets import transforms as DINO_T

CONFIG = REPO / 'config/DINO/DINO_4scale.py'
parser = get_args_parser()
args = parser.parse_args(['--config_file', str(CONFIG)])
cfg = SLConfig.fromfile(str(CONFIG))
cfg.merge_from_dict({'num_expansion': 3, 'enc_scale': 1})
for k,v in cfg._cfg_dict.to_dict().items():
    if hasattr(args, k):
        raise ValueError(f'중복된 설정 키: {k}')
    setattr(args, k, v)
args.device = 'cuda'
args.use_ema = False
args.debug = False
args.key_aware = True  # 체크포인트와 strict=True 일치 확인됐던 구조
assert (args.num_expansion, args.enc_scale, args.key_aware) == (3, 1, True)

if 'model' in globals():
    del model
    gc.collect(); torch.cuda.empty_cache()
model, criterion, postprocessors = build_model_main(args)
# 공식 GitHub 릴리스 체크포인트만 불러오세요.
checkpoint = torch.load(LOCAL_CKPT, map_location='cpu', weights_only=False)
model.load_state_dict(checkpoint['model'], strict=True)
del checkpoint, criterion
gc.collect()
model = model.to('cuda').eval()

preprocess = DINO_T.Compose([
    DINO_T.RandomResize([480], max_size=854),
    DINO_T.ToTensor(),
    DINO_T.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
])
print('PASS: 모델 생성 + strict 가중치 로딩 + GPU')
print('GPU 할당:', round(torch.cuda.memory_allocated()/1024**3, 2), 'GiB')

## 9. 프레임 선택 · 추론 · JSONL 안전 재개 함수

In [ ]:
import itertools
from PIL import Image


def box_iou(a, b):
    x1, y1 = max(a[0],b[0]), max(a[1],b[1])
    x2, y2 = min(a[2],b[2]), min(a[3],b[3])
    inter = max(0.,x2-x1)*max(0.,y2-y1)
    area_a = max(0.,a[2]-a[0])*max(0.,a[3]-a[1])
    area_b = max(0.,b[2]-b[0])*max(0.,b[3]-b[1])
    union = area_a+area_b-inter
    return inter/union if union>0 else 0.


def nms(dets, threshold=NMS_IOU):
    kept=[]
    for d in sorted(dets,key=lambda x:-x['confidence']):
        if all(box_iou(d['bbox'],p['bbox']) < threshold for p in kept):
            kept.append(d)
    return kept


def inspect_and_recover_jsonl(path, video_id, timestamps):
    if not path.exists(): return 0
    n, valid_end = 0, 0
    with path.open('rb') as f:
        while True:
            raw = f.readline()
            if not raw: return n
            try:
                assert raw.endswith(b'\n'), '미완성 JSONL 행'
                item = json.loads(raw.decode('utf-8'))
                assert n < len(timestamps), '프레임 수 초과'
                expected = timestamps[n]
                assert item['video_id'] == video_id, '다른 video_id가 기록됨'
                assert item['requested_timestamp_ms'] == expected, '요청 timestamp 불일치'
                assert abs(int(item['actual_timestamp_ms'])-expected) <= MAX_PTS_ERROR_MS, 'PTS 오차'
                assert item.get('record_type') == 'detection_experiment', '기록 유형 불일치'
                assert isinstance(item['detections'],list), 'detections 형식 이상'
            except Exception as exc:
                # 비정상 종료로 마지막 한 줄만 미완성인 경우에 한해 복구합니다.
                if f.readline():
                    raise RuntimeError(f'{path}: {n+1}행 중간 손상, 수동 확인 필요: {exc}') from exc
                # 이미 완결된 JSON인데 내용이 틀린 경우는 자동으로 삭제하지 않습니다.
                if raw.endswith(b'\n'):
                    raise RuntimeError(f'{path}: {n+1}행 내용/ID 불일치: {exc}') from exc
                backup = path.with_name(path.name + f'.incomplete_backup_{int(time.time())}')
                shutil.copy2(path,backup)
                with path.open('r+b') as w:
                    w.truncate(valid_end)
                print('불완전한 마지막 JSONL 행만 복구, 백업:', backup)
                return n
            n += 1
            valid_end = f.tell()




def iter_pts_nearest(job, start_idx):
    """순차 디코딩. 각 요청마다 '첫 PTS > 요청' 경계까지의 프레임에서 최근접 선택.
    비단조 PTS의 경우에도 원본 노트북 방식의 디코드 순서 기반 정책을 보존합니다.
    """
    from collections import deque
    path = job['path']
    cache = deque()
    first_pts = None
    with av.open(str(path)) as container:
        stream = container.streams.video[0]
        frame_iter = enumerate(container.decode(stream))
        exhausted = False
        for requested in job['timestamps'][start_idx:]:
            while not exhausted and not any(x[0] > requested for x in cache):
                try: idx, frame = next(frame_iter)
                except StopIteration:
                    exhausted = True
                    break
                if frame.pts is not None and frame.time_base is not None:
                    pts = float(frame.pts * frame.time_base)
                    if first_pts is None: first_pts = pts
                    actual = job['offset_ms'] + round((pts - first_pts)*1000)
                    source = 'pts'
                else:
                    rate = stream.average_rate
                    if rate is None or float(rate) <= 0:
                        raise RuntimeError('PTS가 없고 FPS도 없어 시각 추정 불가')
                    actual = job['offset_ms'] + round(idx * 1000 / float(rate))
                    source = 'estimated'
                if actual >= requested - MAX_PTS_ERROR_MS:
                    cache.append((actual, idx, source, frame))
            cache = deque(x for x in cache if x[0] >= requested - MAX_PTS_ERROR_MS)
            if not cache:
                raise RuntimeError(f"{job['video_id']}: {requested}ms 근처 프레임 없음")
            actual, idx, source, selected = min(cache, key=lambda x: (abs(x[0]-requested), x[0]))
            if abs(actual-requested) > MAX_PTS_ERROR_MS:
                raise RuntimeError(f"{job['video_id']}: 요청 {requested}ms, 실제 {actual}ms (오차 초과)")
            yield dict(requested_timestamp_ms=requested, actual_timestamp_ms=actual,
                frame_index=idx, time_source=source, source_offset_ms=job['offset_ms'],
                rgb=selected.to_ndarray(format='rgb24'))


def run_one_video(job):
    video_id, outfile = job['video_id'],job['output']
    timestamps,total=job['timestamps'],job['total']
    print('\n영상:',video_id,'| 목표:',total,'| 출력:',outfile,flush=True)
    start=inspect_and_recover_jsonl(outfile,video_id,timestamps)
    print('이전 정상 프레임:',start,'/',total)
    if start==total: return
    assert next(model.parameters()).device.type=='cuda' and not model.training
    assert 'bbox' in postprocessors
    iterator = iter_pts_nearest(job,start)
    # 최소 1프레임 검사: 프레임 크기·시각 조건
    from PIL import Image
    started=time.perf_counter()
    with outfile.open('a',encoding='utf-8') as f:
        for j,fr in enumerate(iterator,start=start):
            assert fr['requested_timestamp_ms']==timestamps[j]
            image=Image.fromarray(fr['rgb'])
            assert (image.width,image.height)==(job['width'],job['height'])
            tensor,_=preprocess(image,None)
            tensor=tensor.to('cuda')
            torch.cuda.synchronize()
            t0=time.perf_counter()
            with torch.inference_mode():
                pred=model([tensor])
                sizes=torch.tensor([[image.height,image.width]],device='cuda')
                result=postprocessors['bbox'](pred,sizes)[0]
            torch.cuda.synchronize()
            latency=time.perf_counter()-t0
            dets=[]
            for score,cls,bb in zip(result['scores'],result['labels'],result['boxes']):
                s,c=float(score),int(cls)
                if not np.isfinite(s) or s<CONF_THRESHOLD or c not in CLASSES: continue
                bbox=[float(v) for v in bb]
                if not all(np.isfinite(v) for v in bbox): continue
                x1,y1,x2,y2=bbox
                bbox=[max(0.,min(float(image.width),x1)), max(0.,min(float(image.height),y1)),
                    max(0.,min(float(image.width),x2)), max(0.,min(float(image.height),y2))]
                if bbox[0]>=bbox[2] or bbox[1]>=bbox[3]: continue
                dets.append(dict(class_id=c,class_name=CLASSES[c],confidence=s,bbox=bbox))
            dets=nms(dets)
            record=dict(schema_version='cctv-3.0',record_type='detection_experiment',
                video_id=video_id,requested_timestamp_ms=fr['requested_timestamp_ms'],
                actual_timestamp_ms=fr['actual_timestamp_ms'],timestamp_ms=fr['actual_timestamp_ms'],
                frame_index=fr['frame_index'],width=image.width,height=image.height,
                time_source=fr['time_source'],source_offset_ms=fr['source_offset_ms'],
                inference_seconds=latency,detections=dets)
            f.write(json.dumps(record,ensure_ascii=False)+'\n')
            f.flush()
            del pred,result,tensor
            if (j+1)%25==0 or j+1==total:
                print(f'{video_id} {j+1}/{total} | {len(dets)} persons | {latency:.3f}s | '
                    f'{(time.perf_counter()-started)/60:.1f}min',flush=True)
    n=inspect_and_recover_jsonl(outfile,video_id,timestamps)
    assert n==total, f'{video_id}: 불완료 {n}/{total}'
    print('PASS:',video_id,n,flush=True)

## 10. 탐지 실행 (SMOKE_TEST=True면 영상당 3프레임)

In [ ]:
JOBS = []
for j in VIDEO_CONFIGS:
    job = dict(j)
    if SMOKE_TEST:
        job['timestamps'] = job['timestamps'][:SMOKE_FRAMES]
        job['total'] = len(job['timestamps'])
        smoke_dir = OUT / 'smoke_test'
        smoke_dir.mkdir(parents=True, exist_ok=True)
        job['output'] = smoke_dir / f'detections_{job["video_id"]}_pts.jsonl'
    JOBS.append(job)
for job in JOBS:
    run_one_video(job)

## 11. 출력 검증 및 요약

In [ ]:
summary=[]
for job in JOBS:
    n=inspect_and_recover_jsonl(job['output'],job['video_id'],job['timestamps'])
    if n != job['total']:
        raise RuntimeError(f"{job['video_id']}: {n}/{job['total']} 줄만 처리됨")
    detections=0
    with job['output'].open(encoding='utf-8') as f:
        for line in f:
            r=json.loads(line)
            assert r['schema_version']=='cctv-3.0'
            assert r['video_id']==job['video_id']
            for d in r['detections']:
                assert d['class_id']==1 and d['class_name']=='person'
                assert 0 <= d['bbox'][0] < d['bbox'][2] <= r['width']
                assert 0 <= d['bbox'][1] < d['bbox'][3] <= r['height']
                detections += 1
    summary.append(dict(video_id=job['video_id'],frames=n,detections=detections,
        path=str(job['output']),smoke_test=SMOKE_TEST))
print(json.dumps(summary,ensure_ascii=False,indent=2))
report=OUT / ('smoke_test_report.json' if SMOKE_TEST else 'multivideo_detection_report.json')
report.write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
print('PASS:',report)

## 12. GitHub 제출 및 연동 주의사항

- 이 노트북은 **Lite-DETR 탐지 JSONL 생성**까지만 수행합니다. ByteTrack 및 B3/B4 생성은 기존 B Python 모듈로 별도 실행하세요.
- 처음에는 `SMOKE_TEST=True`로 실행하고, 전체 추론 시 설정 셀의 `SMOKE_TEST=False`로 변경한 뒤 위에서부터 다시 실행하세요. **기존 T3 파일은 덮어쓰지 않습니다.**
- `VIDEO_IDS=None`은 폴더에 있는 모든 영상 대상입니다. 1개만 시험하려면 `VIDEO_IDS=['영상파일명_확장자제외']`를 사용하세요.
- 같은 파일명이 다른 S3 Key에 존재할 수 있으니 다운로드 전에 파일명을 확인하세요.
- 미검증: 신규 영상 전체 추론 및 ByteTrack/B3-B4 통합, PTS 정합성 전수 검증.
